# Step 1 — Embeddings & Semantic Search

**Goal:** see how a computer can tell that two sentences *mean* the same thing, even when they share no words.

An **embedding model** turns a sentence into a list of numbers (a **vector**). Sentences with similar meaning get vectors that point in a similar direction. We measure "how similar" with **cosine similarity**:

- `1.0` → same meaning
- `0.0` → unrelated

At the end we compare this with a naive **keyword search** that only counts shared words.

### Setup
Run this cell first. It lets the notebook import the lab's own code (`config.py`, `rag/`, `agent/`, `llm.py`) from the folder above `notebooks/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
print("Project folder:", ROOT)

## 1. Load the embedding model
`all-MiniLM-L6-v2` is a small neural network (about 90 MB, downloaded the first time). Its only job is **text → 384 numbers**. It is *not* the chat model.

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer

import config

model = SentenceTransformer(config.EMBEDDING_MODEL, device="cpu")
print("Loaded:", config.EMBEDDING_MODEL)

## 2. A tiny knowledge base
Four HR policy lines mixed with sentences about cricket and cooking.

In [ ]:
SENTENCES = [
    "Employees get 5 working days of Marriage Leave.",
    "Sick Leave of more than 2 consecutive days requires a medical certificate.",
    "Employees earn 1.5 days of Earned Leave for every completed month.",
    "Comp Off must be used within 60 days of the date it was earned.",
    "Virat Kohli scored a brilliant century in the test match.",
    "India won the cricket world cup final by six wickets.",
    "Add the onions to the pan and fry them until golden brown.",
    "Biryani tastes best when the rice is cooked with whole spices.",
]

## 3. Turn every sentence into a vector
`model.encode()` returns a matrix: one row per sentence, 384 columns.

In [ ]:
vectors = model.encode(SENTENCES)
print("Shape:", vectors.shape, "→", len(SENTENCES), "sentences ×", vectors.shape[1], "numbers")
print("First 8 numbers of sentence 1:", np.round(vectors[0][:8], 4))

## 4. Two ways to compare: meaning vs words

$$\cos(\theta) = \frac{a \cdot b}{\lVert a \rVert \, \lVert b \rVert}$$

`keyword_overlap` is a naive "search engine": the fraction of query words that also appear in the sentence.

In [ ]:
import re

def cosine_similarity(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def keyword_overlap(query, sentence):
    words = lambda s: set(re.findall(r"[a-z]+", s.lower()))
    q = words(query)
    return len(q & words(sentence)) / max(1, len(q))

## 5. Ask a question
✏️ **Try it:** change `QUERY` and run the next two cells again.

In [ ]:
QUERY = "How many days off can I take for my wedding?"

In [ ]:
query_vector = model.encode(QUERY)
rows = [(cosine_similarity(query_vector, v), keyword_overlap(QUERY, s), s) for v, s in zip(vectors, SENTENCES)]
rows.sort(key=lambda r: r[0], reverse=True)

print(f"{'rank':<5}{'cosine':>8}{'keyword':>9}   sentence")
for rank, (cos, kw, sentence) in enumerate(rows, start=1):
    print(f"{rank:<5}{cos:>8.3f}{kw:>9.2f}   {sentence}")

## What did we learn?
- The **Marriage Leave** sentence ranks first by cosine similarity, although it never says "wedding" or "days off".
- Keyword search scores it low, because it only matches exact words.
- Embeddings match on **meaning**. This is the foundation of the retrieval step in RAG.

**Discuss:** try `"Who won the match?"` or `"How do I make rice tasty?"`. Do the scores make sense?